# PhyloDater 快速入门

> **语言：**[English (canonical)](01_quickstart.ipynb) · 中文（本文档）

十分钟走一遍真实运行：先用命令行，再用 Python API，最后跑随包发布的"带真值"精度核对。

**开始之前**：PhyloDater 是编排层，不自带定年引擎。请先安装至少一个引擎，并检查当前环境里
到底有哪些（本笔记本最后一节就是做这件事的）。逐个引擎的安装说明（含已从 conda / PyPI 下架
的引擎）见 [docs/EXTERNAL_TOOLS_CN.md](../docs/EXTERNAL_TOOLS_CN.md)。
请从已激活环境的 kernel 运行本笔记本（其 `PATH` 需包含这些引擎），否则对应方法会报
`Environment not available` 并跳过。

请在 `tutorials/` 目录下运行各单元格（代码中的路径以此为基准）。

## 1. Inputs

Three files are needed: a tree, an alignment and a calibration configuration. The
repository ships examples, so nothing has to be invented here:

- `examples/tree.nwk`, `examples/alignment.fasta`, `examples/calibrations.yaml`
- `test-data/benchmark/…` — the same three kinds of file, plus `truth.yaml` with
  known divergence times used for the accuracy check in §4.

In [ ]:
from pathlib import Path

repo = Path().resolve().parent          # run this notebook from tutorials/
for rel in ["examples/tree.nwk", "examples/alignment.fasta",
            "examples/calibrations.yaml",
            "test-data/benchmark/tree.nwk", "test-data/benchmark/truth.yaml"]:
    p = repo / rel
    print(f"{'ok ' if p.exists() else 'MISSING'} {rel}")
print()
print((repo / "examples" / "tree.nwk").read_text().strip())
print()
print((repo / "examples" / "calibrations.yaml").read_text().strip()[:400])

## 2. Run a dating analysis with the CLI

`phylodater dating` is the entry point. `-t/-s/-c` are the three inputs, `-o` the
output directory, `--method` one of the seven supported engines, and `--seed` makes
stochastic engines repeatable. PATHd8 is deterministic and fast, so it makes a good
first run.

In [ ]:

import subprocess, sys, shutil
from pathlib import Path

repo = Path().resolve().parent           # run this notebook from tutorials/
out = Path.cwd() / "quickstart_output_pathd8"     # inside tutorials/
shutil.rmtree(out, ignore_errors=True)

cmd = [
    sys.executable, "-m", "phylodater", "dating",
    "-t", str(repo / "examples" / "tree.nwk"),
    "-s", str(repo / "examples" / "alignment.fasta"),
    "-c", str(repo / "examples" / "calibrations.yaml"),
    "-o", str(out),
    "--method", "pathd8",
    "--seed", "42",
]
print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=repo)
print("exit code:", result.returncode)
print(result.stdout[-1500:])


### What was written

Every method produces the same standardised set: a dated tree, a `comparison_table.tsv`
(one row per node, one column block per method), a `summary.txt`, a `comparison_plot.png`
and a `runtime_metadata.json` with timing, warnings and the **engine version** that
produced the numbers.

In [ ]:

import csv
from pathlib import Path

out = Path.cwd() / "quickstart_output_pathd8"
print("files written by the run:")
for p in sorted(out.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(out))

table = out / "comparison_table.tsv"
if table.exists():
    rows = list(csv.DictReader(table.open(), delimiter="\t"))
    print("\nnode ages reported by PATHd8:")
    for r in rows[:8]:
        mean = r.get("pathd8_mean", "")
        print(f"  {r['Node']:<18} {mean}")


## 3. The Python API

The same pipeline is available in-process, which is what you want inside a larger
workflow. `PipelineConfig(methods=[...])` accepts several engines and runs them in
parallel.

In [ ]:

import sys
from pathlib import Path

repo = Path().resolve().parent            # run this notebook from tutorials/
sys.path.insert(0, str(repo))             # no-op once PhyloDater is pip-installed

from phylodater import PhylogeneticTree, CalibrationLoader
from phylodater.core import DatingPipeline, PipelineConfig

import shutil
_LIB_OUT = Path.cwd() / "quickstart_output_library"      # inside tutorials/
shutil.rmtree(_LIB_OUT, ignore_errors=True)

tree = PhylogeneticTree.from_file(repo / "examples" / "tree.nwk")
calibrations = CalibrationLoader().load(repo / "examples" / "calibrations.yaml")
print(f"{len(tree.tip_names)} tips, {len(calibrations)} calibrations")

config = PipelineConfig(
    methods=["pathd8"],
    output_dir=_LIB_OUT,
    seed=42,
    save_intermediates=False,
    enable_checkpoint=False,
)
pipeline = DatingPipeline(config)
results = pipeline.run(tree, repo / "examples" / "alignment.fasta", calibrations)

for name, res in results.items():
    print(f"{name}: {len(res.node_ages)} node ages, "
          f"root = {max((a.mean_age for a in res.node_ages.values()), default=float('nan')):.2f} Ma")


## 4. Check accuracy against ground truth

"Did it run?" is not the interesting question — "is the chronogram right?" is.
`test-data/benchmark/` is a strict-clock JC69 simulation whose true node ages are
recorded in `truth.yaml`, so each method can be scored numerically.
`scripts/run_reference_benchmark.py` does that, checking completion, tip
completeness, topology preservation, chronology, calibration honouring and accuracy.

Reading a real fossil calibration as ground truth would be circular — the method is
fed those very numbers — which is why the reference is simulated. Design and gates:
[docs/TESTING.md](../docs/TESTING.md); measured values:
[docs/TEST_RESULTS.md](../docs/TEST_RESULTS.md).

In [ ]:

import shutil, subprocess, sys
from pathlib import Path

repo = Path().resolve().parent
out = Path.cwd() / "quickstart_benchmark"               # inside tutorials/
shutil.rmtree(out, ignore_errors=True)
shutil.rmtree(out, ignore_errors=True)

# Score two methods against the simulated ground truth shipped with the package.
result = subprocess.run(
    [
        sys.executable, str(repo / "scripts" / "run_reference_benchmark.py"),
        "--dataset", str(repo / "test-data" / "benchmark"),
        "--output", str(out),
        "--methods", "pathd8", "lsd2",
    ],
    capture_output=True, text=True, cwd=repo,
)
print(result.stdout[-2500:] or result.stderr[-2500:])


## 5. Which methods can run here?

Tests and the CLI skip engines that are not installed rather than pretending to pass,
so it is worth checking the environment first.

In [ ]:

import importlib.util, shutil

from phylodater.core import DatingMethodRegistry

backend = {
    "mcmctree": ["mcmctree", "baseml"],
    "lsd2": ["iqtree2", "iqtree"],
    "pathd8": ["PATHd8", "pathd8"],
    "treepl": ["treepl", "treePL"],
    "r8s": ["r8s"],
    "wlogdate": ["launch_wLogDate.py"],
    "mdcat": ["mdcat", "md_cat.py"],
}
python_backend = {
    "r8s": "pyr8s",
    "wlogdate": "logdate",
    "mdcat": "emd",
}

print(f"{'method':<10} {'in registry':<12} backend found")
for method in DatingMethodRegistry.list_methods():
    binary = any(shutil.which(b) for b in backend.get(method, []))
    package = importlib.util.find_spec(python_backend.get(method, "")) is not None \
        if method in python_backend else False
    print(f"{method:<10} {'yes':<12} {'yes' if (binary or package) else 'no  (tests skip)'}")


## 6. Where to read on

- [docs/TESTING.md](../docs/TESTING.md) — test methodology and acceptance gates
- [docs/TEST_RESULTS.md](../docs/TEST_RESULTS.md) — measured results for all seven engines
- [docs/EXTERNAL_TOOLS.md](../docs/EXTERNAL_TOOLS.md) — engine installation and provenance
- [docs/ERROR_CODES.md](../docs/ERROR_CODES.md) — exit codes and error classes
- [docs/ARCHITECTURE.md](../docs/ARCHITECTURE.md) — how the package is put together
- `phylodater dating --help` — every option, including `--method-args` for
  engine-specific parameters

### Troubleshooting

| symptom | cause and fix |
|---------|---------------|
| `Environment not available` for one method | that engine is missing from `PATH`; install it ([docs/EXTERNAL_TOOLS.md](../docs/EXTERNAL_TOOLS.md)) or choose another `--method` |
| `ERROR: File not found LG` from LSD2 | an amino-acid model was requested for a nucleotide alignment; unset `model` (auto) or pass `--method-args model=GTR+G` |
| a warning naming one of your calibrations | the constraint could not be expressed by that engine (semantic degradation) — read the warning, it states exactly what the engine was given |
| MD-Cat takes hours | you enabled `ci_nboots`; it defaults to `0` because a single bootstrap replicate is expensive |